In [ ]:
## выполняется в google collab

В целом уже предобученные модели на отложенной выборке показывают хорошие результаты   
(например 'blanchefort/rubert-base-cased-sentiment-rurewiews' дает 0.87777).   
Однако она возвращает 3 варианта тональности, включая нейтральную.   

Возьму модели и дообучу их на 1-2 эпохах. Т.к. у меня в обучающей выборке нет нейтральной   
тональности, то при дообучении модель будет стремиться минимизировать вероятность   
нейтральной тональности. Хотя возвращать будет по-прежнему 3.   
Буду считать: если вероятность позитивного больше, чем негативного -   
отзыв позитивный. В противном случае - негативный. Игнорируя нейтральную)

In [ ]:
import codecs
import json
import os
from pathlib import Path
import pickle as pkl
import random
import re
from typing import List

from google.colab import drive, files
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
os.environ['KAGGLE_USERNAME'] = 'user_name'
os.environ['KAGGLE_KEY'] = 'api_key'

# Подключаю гугл диск с данными
drive.mount('/content/drive')

PATH_DATA = os.path.join(Path.cwd(), 'drive', 'MyDrive')
PATH_SUBM = os.path.join(Path.cwd(), 'submissions')

In [1]:
!kaggle competitions download -c morecomplicatedsentiment

In [ ]:
!pip install transformers

In [ ]:
#!pip install datasets

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import BertTokenizerFast
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments

#from datasets import load_metric

## Загружаю и предобрабатываю собранные данные

In [ ]:
#clean_text = lambda x:' '.join(re.sub('\n|\r|\t|[^а-яa-z]', ' ', x.lower()).split())
clean_text = lambda x:' '.join(re.sub('\n|\r|\t|[^а-я]', ' ', x.lower()).split())
set_tone = lambda x: 1 if int(x) >= 4 else 2

In [ ]:
df = pd.read_csv(os.path.join(PATH_DATA,'ru_train.csv'))

df['clean_text'] = df.review.map(clean_text)
df['tone'] = df.rating.map(set_tone)

# перемешиваю отзывы
df = df.sample(frac=1).reset_index(drop=True)
df.shape, df.head()

Подготавливаю тренировочную и тестовую выборки

In [ ]:
train, test, train_lbl, test_lbl = train_test_split(df.clean_text, df.tone,
                                                    test_size = 0.2,
                                                    stratify = df.tone,
                                                    random_state = 354274,
                                                   )
train.shape, test.shape, train_lbl.shape, test_lbl.shape

In [ ]:
train_lbl.value_counts(), test_lbl.value_counts()

### Попробую несколько моделей, обученных на не отзывах.

In [ ]:
#task='sentiment'
#PRE_TRAINED_MODEL_NAME = f"blanchefort/rubert-base-cased-{task}"

#PRE_TRAINED_MODEL_NAME = 'blanchefort/rubert-base-cased-sentiment-rusentiment'
PRE_TRAINED_MODEL_NAME = 'blanchefort/rubert-base-cased-sentiment-rurewiews'

MODEL_FOLDER = 'ru-blanchefort-rurewiews2'

MAX_LENGTH = 512

Загружаю токенайзер, переводящий данные в embeddings, модель.   
Перевожу данные в torch.tensor, необходимый для обучения.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

train_tokens = tokenizer(list(train.values), truncation=True, padding=True, max_length=MAX_LENGTH)
test_tokens  = tokenizer(list(test.values),  truncation=True, padding=True, max_length=MAX_LENGTH)

In [ ]:
class TonalityDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels: List[str]) -> None:
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx: int) -> int:
        '''
        Получение метки класса из тензора
        args:
            idx - индекс требуемой метки класса
        return:
            int - метка класса
        '''
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor([self.labels[idx]])
        
        return item

    def __len__(self) -> int:
        return len(self.labels)


# 0: NEUTRAL
# 1: POSITIVE
# 2: NEGATIVE
def compute_metrics(pred) -> Dict:
    '''
    Расчет метрики roc-auc для предсказанных и истинных значениях классов
    '''
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # roc_auc_score используя sklearn
    rocauc = roc_auc_score(labels, preds)

    return {
        'roc-auc': rocauc,
    }

@torch.no_grad()
def predict(text: List[str]) -> List[float, float]:
    '''
    Предсказание метки класса для отзывов
    args:
        text - тексты отзывов
    return:
        вероятности классов
    '''
    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')
    outputs = model(**inputs)
    predicted = torch.nn.functional.softmax(outputs.logits, dim=1)
    #predicted = torch.argmax(predicted, dim=1).numpy()
    return predicted


In [ ]:
train_dataset = TonalityDataset(train_tokens, train_lbl.values)
test_dataset  = TonalityDataset(test_tokens,  test_lbl.values)

In [ ]:
training_args = TrainingArguments(
    output_dir=f'/content/drive/MyDrive/models/{MODEL_FOLDER}/results',  # output directory
    num_train_epochs=1,             # total number of training epochs
    #per_device_train_batch_size=16,  # batch size per device during training
    #per_device_eval_batch_size=20,   # batch size for evaluation
    per_device_train_batch_size=8,  # batch size per device during training
    per_device_eval_batch_size=16,  # batch size for evaluation
    warmup_steps=300,               # number of warmup steps for learning rate scheduler
    weight_decay=0.01,              # strength of weight decay
    #logging_dir=f'/content/drive/MyDrive/models/{MODEL_FOLDER}/logs', # directory for storing logs
    #logging_first_step  = True, 
    load_best_model_at_end=True,    # load the best model when finished training (default metric is loss)
    # but you can specify `metric_for_best_model` argument to change to accuracy or other metric
    #metric_for_best_model = compute_metrics,
    greater_is_better=True,       # if use roc-auc inside 'metric_for_best_model'
    logging_steps=50,               # log & save weights each logging_steps
    evaluation_strategy='steps',    #'epoch;' # evaluate each `logging_steps`

    seed = 112,
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(PRE_TRAINED_MODEL_NAME,)
model.classifier

In [ ]:
# заменяю голову модели на выход для 2х классов
#model.classifier = torch.nn.Linear(in_features=768, out_features=2, bias=True)
#model.classifier

In [ ]:
trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset,           # evaluation dataset
    compute_metrics=compute_metrics,     # the callback that computes metrics of interest
)

Считаю на cpu. В зависимости от нагрузки colab может не успеть просчитать все.  
Так что сохраняю промежуточные состояния на g.drive и, после сброса colab сессии,   
вновь запускаю подсчет.

In [ ]:
trainer.train(resume_from_checkpoint = True)

In [ ]:
model.save_pretrained(os.path.join('/content/drive/MyDrive/models/', f'{submname}'))

Делаю сабмит.

In [ ]:
with codecs.open(os.path.join('.', 'test.csv'), encoding='utf-8') as fd:
    data = fd.read()

In [ ]:
# разделяю на отзывы. удалаяю <review> в начале и конце каждого отзыва
data = data.lower().split('</review>\n\n<review>')
data[0] = data[0][8:]
data[-1] = data[-1][:-11]

len(data)

In [ ]:
data = list(map(clean_text, data))

### Получившиеся данные

In [ ]:
data[random.randrange(100)]

In [ ]:
np.mean([len(el.split()) for el in data])

In [ ]:
subm = pd.read_csv(os.path.join('.', 'sample_submission.csv'))

In [ ]:
%%time
pred = predict(data)

# по вероятностям классов разделяю на позитивные и негативные отзывы
for idx in subm.index:
    if pred[idx][1].item() > pred[idx][2].item():
        subm.loc[idx, 'y'] = 'pos'
    else:
        subm.loc[idx, 'y'] = 'neg'

subm.y.value_counts()

Сохраним в сабмит

In [2]:
submname = 'ru-blanchefort-rurewiew_tuned_1epochs'
subm.to_csv(os.path.join('.', f'{submname}.csv'), index = False)

files.download(f'{submname}.csv') 

NameError: name 'subm' is not defined

результаты:   
XXXXXXX ru-blanchefort-rurewiew_stem   

0.72222 ru-blanchefort-sentiment   
0.78888 ru-blanchefort-rusentiment   
0.81111 ru-blanchefort-rurewiew_lemm   
0.85555 ru-blanchefort-rurewiew_nosw   
0.87777 ru-blanchefort-rurewiew   
0.97777 ru-blanchefort-rurewiew_tuned_1epochs   